<a href="https://colab.research.google.com/github/arauch6363-crypto/pt/blob/main/PT_github_2026_Analysis_clean.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PT Analysis 2026 — Cleaned

Produces per-race `.txt` files and consolidated CSVs for AI analysis.

## 1. Imports

In [ ]:
# Standard library
import sys, time, ast, json, re, warnings
from datetime import datetime, timedelta
from itertools import cycle
from io import StringIO

# Data
import numpy as np
import pandas as pd
import pytz
from tqdm import tqdm
import json

# Stats / ML
from scipy.stats import ttest_ind
from statsmodels.nonparametric.smoothers_lowess import lowess
from textblob import TextBlob

# Scikit-learn
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import (
    OneHotEncoder, LabelEncoder, StandardScaler, MinMaxScaler
)
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import (
    train_test_split, GroupShuffleSplit, GroupKFold, GridSearchCV
)
from sklearn.metrics import mean_squared_error, roc_auc_score, make_scorer
from sklearn.impute import KNNImputer
from sklearn.base import clone

# Plotting
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib import cm
from matplotlib.lines import Line2D
import os


## 2. Mount Drive

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')


## 3. Load Data

In [ ]:
# ── Historical data ──────────────────────────────────────────────────────────
BASE = "/content/drive/MyDrive/PT" if os.path.exists("/content/drive/MyDrive/PT") else "."

runners = (pd.read_parquet(f"{BASE}/runners.parquet")
             .drop_duplicates(subset=['horseName', 'raceId']))

webTips = (pd.read_parquet(f"{BASE}/webTips.parquet")
             .drop_duplicates(subset=['meetingId', 'raceId']))
webTips = webTips[webTips['text'].notna()]

odds  = pd.read_parquet(f"{BASE}/odds.parquet").drop_duplicates()
races = pd.read_parquet(f"{BASE}/races.parquet").drop_duplicates()


In [ ]:
# ── Today's data ─────────────────────────────────────────────────────────────
BASE = "/content/drive/MyDrive/PT" if os.path.exists("/content/drive/MyDrive/PT") else "."

runners_tdy  = (pd.read_parquet(f"{BASE}/runners_tdy.parquet")
                  .drop_duplicates(subset=['horseName', 'raceId']))
webTips_tdy  = (pd.read_parquet(f"{BASE}/webTips_tdy.parquet")
                  .drop_duplicates(subset=['meetingId', 'raceId']))
odds_tdy     = pd.read_parquet(f"{BASE}/odds_tdy.parquet").drop_duplicates()
races_tdy    = pd.read_parquet(f"{BASE}/races_tdy.parquet").drop_duplicates()



## 4. Merge Dataframes

In [ ]:
# Merge historical dataframes
runners = runners.merge(
    races[['id_race', 'class', 'type', 'going', 'date', 'totalPrize',
           'distance', 'name_meeting', 'direction', 'rail', 'surface', 'winnerTime']],
    left_on='raceId', right_on='id_race', how='left'
)
runners = runners.merge(
    odds[['raceId', 'horseName', 'referenceOdd', 'liveOdd']],
    on=['raceId', 'horseName'], how='left'
)
runners = runners.merge(
    webTips[['raceId', 'meetingId', 'text']],
    on=['raceId', 'meetingId'], how='left'
)

# Merge today's runners with race details
runners_tdy = runners_tdy.merge(
    races_tdy[['id_race', 'going', 'distance', 'name_race']],
    left_on='raceId', right_on='id_race', how='left'
)


In [ ]:
#runners_tdy = runners_tdy[runners_tdy['meetingName'].isin(["Les Sables-d'Olonne", 'Deauville', 'Vichy'])]
runners_tdy['going'] = np.where(runners_tdy['meetingName'] == "Pau", 'PSF Standard', np.where(runners_tdy['meetingName'] == 'Marseille-Borély', 'Bon souple', 'Bon souple'))

race_going_map = {
    'Prix du Bois Franc': 'PSF Standard',
    'Prix du Canal du Hameau': 'PSF Standard',
    "Prix du Canal des Morfondus": 'PSF Standard',
    "Prix du Canal Saint-Jean": 'PSF Standard',
    "Prix du Canal des Druides":'PSF Standard'
}

# Apply updates where name_race matches keys in the dict
runners_tdy.loc[
    runners_tdy['name_race'].isin(race_going_map.keys()),
    'going'
] = runners_tdy['name_race'].map(race_going_map)

## 5. Helper Functions

In [ ]:
# ── Margin / lengths helpers ─────────────────────────────────────────────────

def build_margin_lookup(runners):
    return {
        (row.raceId, row.ranking): row.margin
        for row in runners.itertuples(index=False)
    }

def margin_to_length_fast(margin, ranking, raceId, margin_lookup):
    if ranking == 1:
        margin_rank2 = margin_lookup.get((raceId, 2), None)
        if margin_rank2 is None:
            return 0
        return -margin_to_length_fast(margin_rank2, 2, raceId, margin_lookup)
    if pd.isnull(margin):
        return 0
    if isinstance(margin, str):
        margin = margin.replace('(', '').replace(')', '')
        margin_map = {
            'DH': 0, 'NEZ': 0.05, 'CTT': 0.1,
            'TETE': 0.2, 'CTE': 0.3, 'ENC': 0.4,
            'LOIN': 99
        }
        if margin in margin_map:
            return margin_map[margin]
        parts = margin.split()
        length = 0.0
        for part in parts:
            if '/' in part:
                num, denom = part.split('/')
                length += float(num) / float(denom)
            else:
                length += float(part)
        return length
    return float(margin)


# ── Distance grouping ─────────────────────────────────────────────────────────

def get_distance_group(m):
    if pd.isna(m):
        return None
    m = int(m)
    if m <= 1000:
        return '0-1000'
    elif m > 3600:
        return '>3600'
    else:
        lower = ((m - 1) // 200) * 200 + 1
        upper = lower + 199
        return f'{lower}-{upper}'


# ── Web tips processing ───────────────────────────────────────────────────────

def rank_horses_by_mention_order(text, horse_names):
    lower_text = text.lower()
    mention_positions = []
    for horse in horse_names:
        pattern = r'\b' + re.escape(horse.lower()) + r'\b'
        match = re.search(pattern, lower_text)
        if match:
            mention_positions.append((match.start(), horse))
    mention_positions.sort()
    return {horse: rank + 1 for rank, (_, horse) in enumerate(mention_positions)}

def process_webtips_simple(webTips, runners):
    all_records = []
    for row in tqdm(webTips.itertuples(), total=len(webTips), desc="Processing tips"):
        race_id = row.raceId
        text = row.text
        horse_names = list(runners[runners['raceId'] == race_id]['horseName'].dropna())
        ranked_horses = rank_horses_by_mention_order(text, horse_names)
        for horse, rank in ranked_horses.items():
            all_records.append({"race_id": race_id, "runner": horse, "webTipRank": rank})
    return pd.DataFrame(all_records)


# ── Prize money ───────────────────────────────────────────────────────────────

def calculate_prize(total_prize, ranking):
    if pd.isna(total_prize) or pd.isna(ranking):
        return 0
    ranking = int(ranking)
    prize_splits = {1: 0.62, 2: 0.22, 3: 0.12, 4: 0.04}
    return total_prize * prize_splits.get(ranking, 0)


# ── Race time parsing ─────────────────────────────────────────────────────────

def parse_race_time(time_str):
    if pd.isna(time_str):
        return None
    time_str = str(time_str).strip()
    pattern_with_minutes = r"(\d+)'(\d+)\"(\d+)"
    pattern_seconds_only = r"'(\d+)\"(\d+)"
    match = re.match(pattern_with_minutes, time_str)
    if match:
        mins, secs, hundredths = match.groups()
        return int(mins) * 60 + int(secs) + int(hundredths) / 10
    match = re.match(pattern_seconds_only, time_str)
    if match:
        secs, hundredths = match.groups()
        return int(secs) + int(hundredths) / 100
    return None


## 6. Rating System

In [ ]:
# ── Rating engine ─────────────────────────────────────────────────────────────

kg_per_length = {
    'VERY SLOW': {'0-1000': 1.2, '1001-1200': 1.1, '1201-1400': 1.0,
                  '1401-1600': 0.95, '1601-1800': 0.9, '1801-2000': 0.85,
                  '2001-2200': 0.8, '2201-2400': 0.75, '2401-2600': 0.7,
                  '2601-2800': 0.65, '2801-3000': 0.6, '3001-3200': 0.55,
                  '3201-3400': 0.5, '3401-3600': 0.45, '>3600': 0.4},
    'SLOW':      {'0-1000': 1.3, '1001-1200': 1.2, '1201-1400': 1.1,
                  '1401-1600': 1.05, '1601-1800': 1.0, '1801-2000': 0.95,
                  '2001-2200': 0.9, '2201-2400': 0.85, '2401-2600': 0.8,
                  '2601-2800': 0.75, '2801-3000': 0.7, '3001-3200': 0.65,
                  '3201-3400': 0.6, '3401-3600': 0.55, '>3600': 0.5},
    'FAST':      {'0-1000': 1.5, '1001-1200': 1.4, '1201-1400': 1.3,
                  '1401-1600': 1.2, '1601-1800': 1.15, '1801-2000': 1.1,
                  '2001-2200': 1.05, '2201-2400': 1.0, '2401-2600': 0.95,
                  '2601-2800': 0.9, '2801-3000': 0.85, '3001-3200': 0.8,
                  '3201-3400': 0.75, '3401-3600': 0.7, '>3600': 0.65},
    'VERY FAST': {'0-1000': 1.6, '1001-1200': 1.5, '1201-1400': 1.4,
                  '1401-1600': 1.3, '1601-1800': 1.25, '1801-2000': 1.2,
                  '2001-2200': 1.15, '2201-2400': 1.1, '2401-2600': 1.05,
                  '2601-2800': 1.0, '2801-3000': 0.95, '3001-3200': 0.9,
                  '3201-3400': 0.85, '3401-3600': 0.8, '>3600': 0.75},
    'PSF':       {'0-1000': 1.4, '1001-1200': 1.3, '1201-1400': 1.2,
                  '1401-1600': 1.1, '1601-1800': 1.05, '1801-2000': 1.0,
                  '2001-2200': 0.95, '2201-2400': 0.9, '2401-2600': 0.85,
                  '2601-2800': 0.8, '2801-3000': 0.75, '3001-3200': 0.7,
                  '3201-3400': 0.65, '3401-3600': 0.6, '>3600': 0.55},
}

def lengths_to_rating(l):
    if l < 0:
        raise ValueError("Length must be non-negative")
    if l >= 6.0:
        return 5.0
    x = l / 6.0
    y_norm = np.log1p(9 * x) / np.log(10)
    return 1.0 + 4.0 * y_norm

def signed_bucket(l_signed):
    magnitude = lengths_to_rating(abs(l_signed))
    return magnitude if l_signed >= 0 else -magnitude

def update_ratings_with_history(df, k_factor=1.0, alpha=0.07, initial_ratings=None):
    ratings = dict(initial_ratings) if initial_ratings else {}
    rating_history = []
    df = df.sort_values(by='date')
    df['prize_quantile'] = df.groupby('raceId')['totalPrize_y'].rank(pct=True)
    race_ids = df['raceId'].unique()
    for race_id in tqdm(race_ids, desc="Processing races"):
        race = df[df['raceId'] == race_id].copy()
        for _, row in race.iterrows():
            h = row['horseId']
            if h not in ratings:
                if 'handicapRatingKg' in row and not pd.isnull(row['handicapRatingKg']):
                    ratings[h] = 30.0 + (row['handicapRatingKg'] - 30.0)
                else:
                    ratings[h] = 30.0
        horses = {
            row['horseId']: {
                'rating': ratings[row['horseId']],
                'lengths_back': row['cumulative_lengths_back'],
                'weight': row['weightKg']
            }
            for _, row in race.iterrows()
        }
        surprises = {h: 0.0 for h in horses}
        n_duels   = {h: 0   for h in horses}
        horse_ids = list(horses.keys())
        for i in range(len(horse_ids)):
            for j in range(i + 1, len(horse_ids)):
                h1, h2 = horse_ids[i], horse_ids[j]
                d1, d2 = horses[h1], horses[h2]
                perf_diff = (d1['rating'] - 0.625 * d1['weight']) - (d2['rating'] - 0.625 * d2['weight'])
                bucket_exp = signed_bucket(perf_diff)
                going    = race.iloc[0].get('going_category', 'FAST') or 'FAST'
                distance = race.iloc[0].get('distance_group')
                multiplier = kg_per_length.get(going.upper(), kg_per_length['FAST']).get(distance, 1.0)
                length_gap = d2['lengths_back'] - d1['lengths_back']
                actual_diff = length_gap * multiplier
                bucket_act = signed_bucket(actual_diff)
                surprise = bucket_act - bucket_exp
                surprises[h1] += surprise
                surprises[h2] -= surprise
                n_duels[h1] += 1
                n_duels[h2] += 1
        for h in horses:
            avg_surprise = surprises[h] / max(1, n_duels[h])
            horse_row = race[race['horseId'] == h].iloc[0]
            prize_quantile = horse_row.get('prize_quantile', 0.5)
            horse_run      = horse_row.get('horse_run', 1)
            prize_bonus    = (prize_quantile - 0.5) * 5 * np.exp(-alpha * horse_run)
            ratings[h] += k_factor * (avg_surprise + prize_bonus)
            rating_history.append({'raceId': race_id, 'horseId': h, 'rating_after_race': ratings[h]})
    history_df = pd.DataFrame(rating_history)
    return df.merge(history_df, on=['raceId', 'horseId'], how='left')


## 7. Transformations & New Columns

In [ ]:
# ── WebTips scoring ───────────────────────────────────────────────────────────
webTips_scored = process_webtips_simple(webTips, runners)
runners = runners.merge(
    webTips_scored, left_on=['raceId', 'horseName'],
    right_on=['race_id', 'runner'], how='left'
)
runners['runners'] = runners.groupby('raceId')['ranking'].transform('max')
runners = runners.sort_values(by='date', ascending=True)
runners['horse_run'] = runners.groupby('horseId').cumcount() + 1
runners['webTipRank_perc'] = (
    (runners['runners'] - runners['webTipRank']) / (runners['runners'] - 1)
).fillna(0)

# ── Lengths back ──────────────────────────────────────────────────────────────
margin_lookup = build_margin_lookup(runners)
runners['lengths_back'] = runners.apply(
    lambda row: margin_to_length_fast(row['margin'], row['ranking'], row['raceId'], margin_lookup),
    axis=1
)
runners['cumulative_lengths_back'] = (
    runners[runners['lengths_back'] >= 0]
    .sort_values(by='ranking')
    .groupby('raceId')['lengths_back']
    .cumsum()
)

# ── Going / distance categories ───────────────────────────────────────────────
going_mapping = {
    "Lourd": "VERY SLOW", "Très lourd": "VERY SLOW", "Collant": "VERY SLOW",
    "Souple": "SLOW",     "Très souple": "SLOW",
    "Bon souple": "FAST", "Bon": "FAST",
    "Léger": "VERY FAST", "Bon léger": "VERY FAST", "Très léger": "VERY FAST",
    "PSF Standard": "PSF", "PSF Lente": "PSF", "PSF Rapide": "PSF",
    None: None
}

for df in [runners, runners_tdy, races_tdy]:
    col = 'going' if 'going' in df.columns else None
    if col:
        df['going_category'] = df[col].map(going_mapping)
        df['distance_group'] = df['distance'].apply(get_distance_group)

# ── Additional runner columns ─────────────────────────────────────────────────
runners['cumulative_lengths_back'] = runners['cumulative_lengths_back'].fillna(0)
runners['pos_perc']   = (runners['runners'] - runners['ranking']) / (runners['runners'] - 1)
runners['place']      = np.where(runners['ranking'] <= 2, 1,
                          np.where((runners['ranking'] == 3) & (runners['runners'] >= 7), 1, 0))
runners['win']        = np.where(runners['ranking'] <= 1, 1, 0)
runners['prizemoney'] = runners.apply(
    lambda row: calculate_prize(row['totalPrize_y'], row['ranking']), axis=1
)
runners['handicapRating_adj'] = runners['handicapRatingKg'] + 0.625 * (55 - runners['weightKg'])
runners['date']       = pd.to_datetime(runners['date'])
runners               = runners.sort_values(by='date')
runners['cumulative_lengths_back'] = pd.to_numeric(runners['cumulative_lengths_back'], errors='coerce')
runners['odds']       = 1 / runners['liveOdd']
runners['odds_sum']   = runners.groupby(['raceId', 'name_meeting'])['odds'].transform('sum')
runners['place_sp']   = (runners['liveOdd'] - 1) / 4 + 1
runners['odds_place'] = 1 / runners['place_sp']

if 'name_meeting' in runners.columns and 'meetingName' not in runners.columns:
    runners = runners.rename(columns={'name_meeting': 'meetingName'})
runners['runners'] = runners.groupby(['raceId', 'date'])['horseName'].transform('count')
runners['pos_perc']  = (runners['runners'] - runners['ranking']) / (runners['runners'] - 1)

# ── draw_unique ───────────────────────────────────────────────────────────────
def make_draw_unique(row, meeting_col):
    parts = [row['draw'], row[meeting_col], row['distance'],
             row.get('direction'), row.get('surface')]  # removed 'rail'
    return ' - '.join(str(x) for x in parts if pd.notnull(x))

runners['draw_unique'] = runners.apply(make_draw_unique, axis=1, meeting_col='name_meeting')

merged_temp = runners_tdy.merge(
    races_tdy[['id_race', 'direction', 'rail', 'surface']],
    left_on='raceId', right_on='id_race', how='left'
)
runners_tdy['draw_unique'] = merged_temp.apply(
    make_draw_unique, axis=1, meeting_col='meetingName'
)


Processing tips: 100%|██████████| 23693/23693 [00:45<00:00, 517.80it/s]


## 8. Build Ratings

In [ ]:
# Load cached state
df_with_ratings = pd.read_parquet(f"{BASE}/df_with_ratings.parquet")

with open(f"{BASE}/ratings_state.json") as f:
    saved_ratings = {int(k): v for k, v in json.load(f).items()}

with open(f"{BASE}/ratings_last_date.txt") as f:
    last_processed_date = f.read().strip()

print(f"📂 Loaded {len(df_with_ratings)} cached rows (up to {last_processed_date})")

# Find new races not yet in the cache
new_runners = runners[
    (runners['date'] >= '2021-01-01') &
    (runners['date'] > last_processed_date) &
    (~runners['raceId'].isin(df_with_ratings['raceId']))
]

if new_runners.empty:
    print("✅ No new races — using cached ratings.")
else:
    print(f"⚙️  Processing {new_runners['raceId'].nunique()} new races ({len(new_runners)} rows)...")

    new_rated = update_ratings_with_history(
        new_runners, k_factor=0.5, initial_ratings=saved_ratings
    )
    new_rated['rating_after_race'] = new_rated['rating_after_race'].round(1)
    new_rated['draw_unique'] = new_rated.apply(
        make_draw_unique, axis=1, meeting_col='name_meeting'
    )

    # Append and save
    df_with_ratings = pd.concat([df_with_ratings, new_rated], ignore_index=True)
    df_with_ratings.to_parquet(f"{BASE}/df_with_ratings.parquet", index=False)

    # Update ratings state
    updated_ratings = (
        new_rated.sort_values('date')
        .groupby('horseId')['rating_after_race']
        .last()
        .to_dict()
    )
    saved_ratings.update({int(k): v for k, v in updated_ratings.items()})
    with open(f"{BASE}/ratings_state.json", "w") as f:
        json.dump({str(k): v for k, v in saved_ratings.items()}, f)

    with open(f"{BASE}/ratings_last_date.txt", "w") as f:
        f.write(str(df_with_ratings['date'].max()))

    print(f"✅ Appended {len(new_rated)} new rows. Total: {len(df_with_ratings)}")

📂 Loaded 252198 cached rows (up to 2026-02-22 00:00:00)
✅ No new races — using cached ratings.


## 9. Output Formatting & Stat Functions

In [ ]:
# ── Formatting helpers ────────────────────────────────────────────────────────

TEXT_COLS = [
    'horseName', 'date', 'comment', 'sex', 'race_summary', 'meetingName',
    'raceType', 'distance', 'going', 'blinkers', 'tongueTie', 'hood',
    'jockeyName', 'trainerName', 'ownerName'
]

def clean_markdown_table(df, add_ranking=True, decimal_places=None, exclude_rank_for_columns=None):
    """Generate clean markdown table optimised for AI comprehension."""
    if df.empty:
        return "(No data available)"
    df_clean = df.copy()
    if decimal_places is not None:
        for col in df_clean.columns:
            if col not in TEXT_COLS:
                try:
                    df_clean[col] = pd.to_numeric(df_clean[col], errors='coerce').round(decimal_places)
                except (ValueError, TypeError):
                    continue
    df_clean = df_clean.fillna('N/A')
    should_rank = add_ranking
    if exclude_rank_for_columns and any(c in df_clean.columns for c in exclude_rank_for_columns):
        should_rank = False
    if should_rank:
        df_clean.insert(0, 'Rank', range(1, len(df_clean) + 1))
    return df_clean.to_markdown(index=False, tablefmt='grid')

def reorder_by_horses(df, horse_order, horse_col='horseName'):
    """Reorder dataframe to match SP-ranked horse order."""
    if df.empty or not horse_order or horse_col not in df.columns:
        return df
    ordered, remaining = [], df.copy()
    for horse in horse_order:
        mask = remaining[horse_col] == horse
        ordered.append(remaining[mask])
        remaining = remaining[~mask]
    if not remaining.empty:
        ordered.append(remaining)
    return pd.concat(ordered, ignore_index=True) if ordered else df


In [ ]:
# ── Preference stats ─────────────────────────────────────────────────────────

def compute_preference_stats(runners, runners_tdy):
    """Compute per-horse stats & t-tests vs all other conditions."""
    results = []
    for _, tdy_row in runners_tdy.iterrows():
        horseId   = tdy_row['horseId']
        horseName = tdy_row['horseName']
        raceId    = tdy_row['raceId']
        going_cat = tdy_row['going_category']
        dist_grp  = tdy_row['distance_group']
        meet_name = tdy_row['meetingName']
        row_stats = {'horseId': horseId, 'horseName': horseName, 'raceId': raceId}

        def aggregate(group):
            return pd.Series({
                'mean_pos': group['pos_perc'].mean(),
                'runs':     len(group),
                'wins':     group['win'].sum(),
                'places':   group['place'].sum()
            })

        categories = {
            'overall':        runners[runners['horseId'] == horseId],
            'going_category': runners[(runners['horseId'] == horseId) & (runners['going_category'] == going_cat)],
            'distance_group': runners[(runners['horseId'] == horseId) & (runners['distance_group'] == dist_grp)],
            'meetingName':    runners[(runners['horseId'] == horseId) & (runners['meetingName'] == meet_name)],
        }
        for cat, group in categories.items():
            agg = aggregate(group)
            row_stats[f'{cat}_mean']   = agg['mean_pos']
            row_stats[f'{cat}_runs']   = agg['runs']
            row_stats[f'{cat}_wins']   = agg['wins']
            row_stats[f'{cat}_places'] = agg['places']
            other_group = runners[runners['horseId'] == horseId] if cat == 'overall' \
                          else runners[(runners['horseId'] == horseId) & (runners[cat] != tdy_row[cat])]
            g_vals = group['pos_perc'].dropna()
            o_vals = other_group['pos_perc'].dropna()
            if (len(g_vals) >= 5 and len(o_vals) >= 5
                    and g_vals.nunique() > 1 and o_vals.nunique() > 1):
                tscore, pval = ttest_ind(g_vals, o_vals, equal_var=False)
            else:
                tscore, pval = None, None
            row_stats[f'{cat}_tscore'] = tscore
            row_stats[f'{cat}_pval']   = pval
        results.append(row_stats)
    return pd.DataFrame(results)


# ── Over/under-performance stats ─────────────────────────────────────────────

def show_race_stats(runners_tdy, runners, race_idx=0):
    """Compute over/under-performance stats for trainer, jockey, owner, sire."""
    race_id  = runners_tdy.raceId.unique()[race_idx]
    race_row = runners_tdy[runners_tdy['raceId'] == race_id]

    def _ou(entity_col, cond_col=None, cond_val=None):
        sub = runners.copy()
        if cond_col and cond_val is not None:
            sub = sub[sub[cond_col] == cond_val]
        grp = (sub.groupby(entity_col)
                  .agg(runs=('raceId','count'), wins=('win','sum'),
                       places=('place','sum'), odds_sum=('odds','sum'))
                  .assign(ae=lambda d: d['wins'] / d['odds_sum'])
                  .reset_index())
        return grp

    rows = []
    for _, r in race_row.iterrows():
        row = {'horseName': r['horseName']}
        for entity, col in [('trainer','trainerName'), ('jockey','jockeyName'),
                             ('owner','ownerName'), ('sire','sireName')]:
            val = r.get(col)
            grp = _ou(col)
            match = grp[grp[col] == val]
            if not match.empty:
                m = match.iloc[0]
                row[f'{entity}_runs']   = m['runs']
                row[f'{entity}_wins']   = m['wins']
                row[f'{entity}_ae']     = round(m['ae'] * 100, 1)
            else:
                row[f'{entity}_runs'] = row[f'{entity}_wins'] = row[f'{entity}_ae'] = None
        rows.append(row)
    return pd.DataFrame(rows)


In [ ]:
def summarize_horse_stats_by_race(runners_tdy_df, races_tdy_df, runners_df, race):
    race_id = runners_tdy_df.raceId.unique()[race]
    if races_tdy_df is not None:
        race_date = pd.to_datetime(races_tdy_df.loc[races_tdy_df['id_race'] == race_id, 'date'].iloc[0])
        race_going_category = runners_tdy_df.loc[runners_tdy_df['raceId'] == race_id, 'going_category'].iloc[0]
        race_distance_group = races_tdy_df.loc[races_tdy_df['id_race'] == race_id, 'distance_group'].iloc[0]
    else:
        raise ValueError("races_tdy_df is required")

    cutoff_date_365 = race_date - pd.Timedelta(days=365)

    per_horse_today = (runners_tdy_df.loc[runners_tdy_df['raceId'] == race_id,
                                         ['horseId', 'horseName', 'sex', 'age', 'tongueTie',
                                          'hood', 'blinkers', 'handicapRatingKg', 'weightKg']]
                      .drop_duplicates())
    per_horse_today['adjusted_handicapRatingKg'] = (
        per_horse_today['handicapRatingKg'] - per_horse_today['weightKg'] + 55
    )

    hist = runners_df.copy()
    hist['date'] = pd.to_datetime(hist['date'], errors='coerce')
    hist = hist.sort_values(['horseId', 'date'])
    hist['run_to_rating'] = (hist.groupby('horseId', group_keys=False)['rating_after_race']
                               .apply(lambda s: s.diff()))
    historical_data_365 = hist[(hist['date'] >= cutoff_date_365) & (hist['date'] < race_date)].copy()
    historical_data_all = hist[hist['date'] < race_date].copy()

    horse_metrics = []
    for _, horse_row in per_horse_today.iterrows():
        horse_id = horse_row['horseId']
        today_weight_kg = horse_row['weightKg']
        horse_hist_365 = historical_data_365[historical_data_365['horseId'] == horse_id].copy()
        horse_hist_all = historical_data_all[historical_data_all['horseId'] == horse_id].copy()

        if horse_hist_365.empty:
            runs_365, wins_365, places_365, rtr = 0, 0, 0, 0.0
        else:
            runs_365 = len(horse_hist_365)
            wins_365 = horse_hist_365['win'].sum()
            places_365 = horse_hist_365['place'].sum()
            rtr = horse_hist_365['run_to_rating'].dropna().mean()
            if pd.isna(rtr): rtr = 0.0
        runs_wins_places_365d = f"{runs_365}/{wins_365}/{places_365}"

        if horse_hist_all.empty:
            runs_all, wins_all, places_all = 0, 0, 0
        else:
            runs_all = len(horse_hist_all)
            wins_all = horse_hist_all['win'].sum()
            places_all = horse_hist_all['place'].sum()
        runs_wins_places = f"{runs_all}/{wins_all}/{places_all}"

        rtr_formatted = str(np.round(rtr, 2)) + ' (' + str(runs_365) + ')'

        if not horse_hist_all.empty:
            last_run = horse_hist_all.iloc[-1]
            last_adjusted_rating = (last_run['rating_after_race'] + 55 - today_weight_kg
                                    if pd.notna(last_run['rating_after_race']) and pd.notna(today_weight_kg)
                                    else None)
            days_since_last_run = (race_date - last_run['date']).days if pd.notna(last_run['date']) else None
        else:
            last_adjusted_rating = None
            days_since_last_run = None

        if not horse_hist_all.empty:
            last_5_ratings = horse_hist_all.tail(5)['run_to_rating'].dropna()
            rtr_last_5 = (', '.join(str(np.round(v, 2)) for v in reversed(last_5_ratings.tolist()))
                          if len(last_5_ratings) > 0 else '-')
        else:
            rtr_last_5 = '-'

        if race_going_category is not None:
            going_data = horse_hist_all[horse_hist_all['going_category'] == race_going_category]
            going_runs = len(going_data['run_to_rating'].dropna()) if not going_data.empty else 0
            rtr_going_category = going_data['run_to_rating'].dropna().mean() if not going_data.empty else 0.0
        else:
            going_runs, rtr_going_category = 0, 0.0

        if race_distance_group is not None:
            distance_data = horse_hist_all[horse_hist_all['distance_group'] == race_distance_group]
            distance_runs = len(distance_data['run_to_rating'].dropna()) if not distance_data.empty else 0
            rtr_distance_group = distance_data['run_to_rating'].dropna().mean() if not distance_data.empty else 0.0
        else:
            distance_runs, rtr_distance_group = 0, 0.0

        horse_metrics.append({
            'horseId': horse_id,
            'horseName': horse_row['horseName'],
            'runs_wins_places_365d': runs_wins_places_365d,
            'runs_wins_places': runs_wins_places,
            'rtr': rtr_formatted,
            'rtr_going_category': rtr_going_category,
            'rtr_distance_group': rtr_distance_group,
            'rtr_last_5': rtr_last_5,
            'adjusted_rar': last_adjusted_rating,
            'days_since_last_run': days_since_last_run,
            'going_runs': going_runs,
            'distance_runs': distance_runs
        })

    horse_metrics_df = pd.DataFrame(horse_metrics)
    horse_metrics_df['rtr_going_category'] = (
        np.round(horse_metrics_df['rtr_going_category'].fillna(0.0), 2).astype(str) +
        ' (' + horse_metrics_df['going_runs'].astype(str) + ')')
    horse_metrics_df['rtr_distance_group'] = (
        np.round(horse_metrics_df['rtr_distance_group'].fillna(0.0), 2).astype(str) +
        ' (' + horse_metrics_df['distance_runs'].astype(str) + ')')

    out = (per_horse_today
           .merge(horse_metrics_df[['horseId', 'runs_wins_places_365d', 'runs_wins_places', 'rtr',
                                    'rtr_going_category', 'rtr_distance_group',
                                    'rtr_last_5', 'adjusted_rar', 'days_since_last_run']],
                  on='horseId', how='left')
           .fillna({'runs_wins_places_365d': '0/0/0', 'runs_wins_places': '0/0/0',
                    'rtr': '0.0 (0)', 'rtr_going_category': '0.0 (0)',
                    'rtr_distance_group': '0.0 (0)', 'rtr_last_5': '-'}))
    out = out[['horseId', 'horseName', 'sex', 'age', 'tongueTie', 'hood', 'blinkers',
               'adjusted_handicapRatingKg', 'adjusted_rar',
               'runs_wins_places_365d', 'runs_wins_places', 'rtr',
               'rtr_going_category', 'rtr_distance_group', 'rtr_last_5',
               'days_since_last_run']].sort_values(by='adjusted_rar', ascending=False)
    for col in ['rtr', 'runs_wins_places_365d', 'runs_wins_places',
                'rtr_going_category', 'rtr_distance_group', 'rtr_last_5']:
        out[col] = out[col].astype(str)
    return out


def get_horse_form_table(runners_tdy_df, races_tdy_df, runners_df, race):
    race_id = runners_tdy_df.raceId.unique()[race]
    if races_tdy_df is not None:
        race_date = pd.to_datetime(races_tdy_df.loc[races_tdy_df['id_race'] == race_id, 'date'].iloc[0])
    else:
        raise ValueError("races_tdy_df is required")

    todays_runners = (runners_tdy_df.loc[runners_tdy_df['raceId'] == race_id,
                                         ['horseId', 'horseName', 'weightKg']]
                      .drop_duplicates())
    todays_weights = dict(zip(todays_runners['horseId'], todays_runners['weightKg']))
    todays_horse_names = dict(zip(todays_runners['horseId'], todays_runners['horseName']))

    hist = runners_df.copy()
    hist['date'] = pd.to_datetime(hist['date'], errors='coerce')
    hist = hist.sort_values(['horseId', 'date'])
    historical_data = hist[hist['date'] < race_date].copy()

    form_records = []
    for _, horse_row in todays_runners.iterrows():
        horse_id = horse_row['horseId']
        horse_name = horse_row['horseName']
        today_weight = horse_row['weightKg']
        horse_form = historical_data[historical_data['horseId'] == horse_id].copy()
        if horse_form.empty:
            continue
        horse_form['adjusted_handicapRatingKg'] = horse_form['handicapRatingKg'] + 55 - today_weight
        horse_form['adjusted_rar'] = horse_form['rating_after_race'] + 55 - today_weight

        for _, form_race in horse_form.iterrows():
            form_race_id = form_race['raceId']
            form_date = form_race['date']
            historical_race_runners = historical_data[
                (historical_data['raceId'] == form_race_id) &
                (historical_data['date'] == form_date)
            ].copy()

            competing_today = []
            for other_horse_id in todays_runners['horseId']:
                if other_horse_id == horse_id:
                    continue
                other_horse_form = historical_race_runners[historical_race_runners['horseId'] == other_horse_id]
                if not other_horse_form.empty:
                    other = other_horse_form.iloc[0]
                    clb_diff = form_race['cumulative_lengths_back'] - other['cumulative_lengths_back']
                    past_adj = form_race['weightKg'] - other['weightKg']
                    today_adj = -(today_weight - todays_weights[other_horse_id])
                    adjusted_diff = clb_diff + past_adj + today_adj
                    competing_today.append(f"{todays_horse_names[other_horse_id]} ({adjusted_diff:.1f})")
            competing_horses_str = ', '.join(competing_today) if competing_today else '-'

            median_rating_change = None
            if not historical_race_runners.empty:
                rating_changes = []
                for _, top_horse in historical_race_runners[historical_race_runners['horseId'] != horse_id].iterrows():
                    top_horse_id = top_horse['horseId']
                    top_race_rating = top_horse['rating_after_race']
                    subsequent = hist[(hist['horseId'] == top_horse_id) & (hist['date'] > form_date)].head(3)
                    if not subsequent.empty:
                        last_rating = subsequent.iloc[-1]['rating_after_race']
                        if pd.notna(top_race_rating) and pd.notna(last_rating):
                            rating_changes.append(last_rating - top_race_rating)
                if rating_changes:
                    median_rating_change = pd.Series(rating_changes).median()

            form_records.append({
                'horseId': horse_id,
                'horseName': horse_name,
                'date': form_race['date'],
                'meetingName': form_race['meetingName'],
                'raceType': form_race.get('raceType'),
                'class': form_race.get('class'),
                'distance': form_race['distance'],
                'going_category': form_race['going_category'],
                'raceTotalPrize': form_race.get('raceTotalPrize') or form_race.get('totalPrize_y'),
                'draw': form_race['draw'],
                'jockey': form_race['jockeyName'],
                'adjusted_handicapRatingKg': form_race['adjusted_handicapRatingKg'],
                'hood': form_race.get('hood'),
                'tongueTie': form_race.get('tongueTie'),
                'blinkers': form_race.get('blinkers'),
                'ranking': form_race['ranking'],
                'runners': form_race['runners'],
                'cumulative_lengths_back': form_race['cumulative_lengths_back'],
                'comment': form_race.get('comment'),
                'liveOdd': form_race.get('liveOdd'),
                'competing_horses_today': competing_horses_str,
                'adjusted_rar': form_race['adjusted_rar'],
                'median_top_half_rating_change': median_rating_change
            })

    form_df = pd.DataFrame(form_records)
    if form_df.empty:
        return pd.DataFrame(columns=['horseId', 'horseName', 'date', 'meetingName', 'raceType',
                                     'class', 'distance', 'going_category', 'raceTotalPrize', 'draw',
                                     'jockey', 'adjusted_handicapRatingKg', 'hood', 'tongueTie',
                                     'blinkers', 'ranking', 'runners', 'cumulative_lengths_back',
                                     'comment', 'liveOdd', 'competing_horses_today',
                                     'adjusted_rar', 'median_top_half_rating_change'])
    return form_df.sort_values(['horseName', 'date'], ascending=[True, False])


def summarize_trainer_stats_by_race(runners_tdy_df, races_tdy_df, runners_df, race):
    race_id = runners_tdy_df.raceId.unique()[race]
    if races_tdy_df is not None:
        race_date = pd.to_datetime(races_tdy_df.loc[races_tdy_df['id_race'] == race_id, 'date'].iloc[0])
        race_type = races_tdy_df.loc[races_tdy_df['id_race'] == race_id, 'type'].iloc[0]
    else:
        raise ValueError("races_tdy_df is required")

    cutoff_date_365 = race_date - pd.Timedelta(days=365)
    cutoff_date_750 = race_date - pd.Timedelta(days=750)

    per_horse_today = (runners_tdy_df.loc[runners_tdy_df['raceId'] == race_id,
                                         ['horseId', 'trainerName', 'meetingName', 'jockeyName', 'ownerName']]
                      .drop_duplicates())

    hist = runners_df.copy()
    hist['date'] = pd.to_datetime(hist['date'], errors='coerce')
    hist = hist.sort_values(['horseId', 'date'])
    hist['run_to_rating'] = (hist.groupby('horseId', group_keys=False)['rating_after_race']
                               .apply(lambda s: s.diff()))
    historical_data_750 = hist[(hist['date'] >= cutoff_date_750) & (hist['date'] < race_date)].copy()

    horse_metrics = []
    for _, horse_row in per_horse_today.iterrows():
        horse_id = horse_row['horseId']
        trainer_name = horse_row['trainerName']
        meeting_name = horse_row['meetingName']
        jockey_name = horse_row['jockeyName']
        owner_name = horse_row['ownerName']

        trainer_hist = hist[(hist['trainerName'] == trainer_name) & (hist['date'] < race_date)].copy()
        trainer_365_data = trainer_hist[trainer_hist['date'] >= cutoff_date_365].copy()

        if trainer_365_data.empty:
            runs, rtr = 0, 0.0
        else:
            runs = len(trainer_365_data)
            rtr = trainer_365_data['run_to_rating'].dropna().mean()
            if pd.isna(rtr): rtr = 0.0
        rtr_formatted = str(np.round(rtr, 2)) + ' (' + str(runs) + ')'
        rtr_last20 = trainer_hist.tail(20)['run_to_rating'].dropna().mean() if not trainer_hist.empty else 0.0

        if race_type is not None:
            type_data = historical_data_750[(historical_data_750['trainerName'] == trainer_name) &
                                             (historical_data_750['type'] == race_type)]
            type_runs = len(type_data['run_to_rating'].dropna()) if not type_data.empty else 0
            rtr_raceType = type_data['run_to_rating'].dropna().mean() if not type_data.empty else 0.0
        else:
            type_runs, rtr_raceType = 0, 0.0

        meeting_data = historical_data_750[(historical_data_750['trainerName'] == trainer_name) &
                                            (historical_data_750['meetingName'] == meeting_name)]
        meeting_runs = len(meeting_data['run_to_rating'].dropna()) if not meeting_data.empty else 0
        rtr_meeting = meeting_data['run_to_rating'].dropna().mean() if not meeting_data.empty else 0.0

        tj_data = historical_data_750[(historical_data_750['trainerName'] == trainer_name) &
                                       (historical_data_750['jockeyName'] == jockey_name)]
        trainer_jockey_runs = len(tj_data['run_to_rating'].dropna()) if not tj_data.empty else 0
        rtr_trainer_jockey = tj_data['run_to_rating'].dropna().mean() if not tj_data.empty else 0.0

        to_data = historical_data_750[(historical_data_750['trainerName'] == trainer_name) &
                                       (historical_data_750['ownerName'] == owner_name)]
        trainer_owner_runs = len(to_data['run_to_rating'].dropna()) if not to_data.empty else 0
        rtr_trainer_owner = to_data['run_to_rating'].dropna().mean() if not to_data.empty else 0.0

        horse_metrics.append({
            'horseId': horse_id, 'trainerName': trainer_name, 'runs': runs,
            'rtr': rtr_formatted, 'rtr_last20': rtr_last20,
            'rtr_raceType': rtr_raceType, 'rtr_meeting': rtr_meeting,
            'rtr_trainer_jockey': rtr_trainer_jockey, 'rtr_trainer_owner': rtr_trainer_owner,
            'type_runs': type_runs, 'meeting_runs': meeting_runs,
            'trainer_jockey_runs': trainer_jockey_runs, 'trainer_owner_runs': trainer_owner_runs
        })

    hm = pd.DataFrame(horse_metrics)
    hm['rtr_last20'] = np.round(hm['rtr_last20'].fillna(0.0), 2)
    for col, run_col in [('rtr_raceType', 'type_runs'), ('rtr_meeting', 'meeting_runs'),
                          ('rtr_trainer_jockey', 'trainer_jockey_runs'), ('rtr_trainer_owner', 'trainer_owner_runs')]:
        hm[col] = np.round(hm[col].fillna(0.0), 2).astype(str) + ' (' + hm[run_col].astype(str) + ')'

    out = (per_horse_today
           .merge(hm[['horseId', 'runs', 'rtr', 'rtr_last20', 'rtr_raceType',
                       'rtr_meeting', 'rtr_trainer_jockey', 'rtr_trainer_owner']], on='horseId', how='left')
           .fillna({'runs': 0, 'rtr': '0.0 (0)', 'rtr_last20': 0.0, 'rtr_raceType': '0.0 (0)',
                    'rtr_meeting': '0.0 (0)', 'rtr_trainer_jockey': '0.0 (0)', 'rtr_trainer_owner': '0.0 (0)'}))
    out = out[['horseId', 'trainerName', 'rtr', 'rtr_last20', 'rtr_raceType',
               'rtr_meeting', 'rtr_trainer_jockey', 'rtr_trainer_owner']].sort_values('rtr', ascending=False)
    out['rtr'] = out['rtr'].astype(str)
    return out


def summarize_jockey_stats_by_race(runners_tdy_df, races_tdy_df, runners_df, race):
    race_id = runners_tdy_df.raceId.unique()[race]
    if races_tdy_df is not None:
        race_date = pd.to_datetime(races_tdy_df.loc[races_tdy_df['id_race'] == race_id, 'date'].iloc[0])
    else:
        raise ValueError("races_tdy_df is required")

    going_category = runners_tdy_df.loc[runners_tdy_df['raceId'] == race_id, 'going_category'].iloc[0]
    cutoff_date_365 = race_date - pd.Timedelta(days=365)
    cutoff_date_750 = race_date - pd.Timedelta(days=750)

    per_horse_today = (runners_tdy_df.loc[runners_tdy_df['raceId'] == race_id,
                                         ['horseId', 'jockeyName', 'meetingName', 'trainerName']]
                      .drop_duplicates())

    hist = runners_df.copy()
    hist['date'] = pd.to_datetime(hist['date'], errors='coerce')
    hist = hist.sort_values(['horseId', 'date'])
    hist['run_to_rating'] = (hist.groupby('horseId', group_keys=False)['rating_after_race']
                               .apply(lambda s: s.diff()))
    historical_data_750 = hist[(hist['date'] >= cutoff_date_750) & (hist['date'] < race_date)].copy()

    horse_metrics = []
    for _, horse_row in per_horse_today.iterrows():
        horse_id = horse_row['horseId']
        jockey_name = horse_row['jockeyName']
        meeting_name = horse_row['meetingName']
        trainer_name = horse_row['trainerName']

        jockey_hist = hist[(hist['jockeyName'] == jockey_name) & (hist['date'] < race_date)].copy()
        jockey_365_data = jockey_hist[jockey_hist['date'] >= cutoff_date_365].copy()

        if jockey_365_data.empty:
            runs, rtr = 0, 0.0
        else:
            runs = len(jockey_365_data)
            rtr = jockey_365_data['run_to_rating'].dropna().mean()
            if pd.isna(rtr): rtr = 0.0
        rtr_formatted = str(np.round(rtr, 2)) + ' (' + str(runs) + ')'
        rtr_last20 = jockey_hist.tail(30)['run_to_rating'].dropna().mean() if not jockey_hist.empty else 0.0

        if going_category is not None:
            going_data = historical_data_750[(historical_data_750['jockeyName'] == jockey_name) &
                                              (historical_data_750['going_category'] == going_category)]
            going_runs = len(going_data['run_to_rating'].dropna()) if not going_data.empty else 0
            rtr_going_category = going_data['run_to_rating'].dropna().mean() if not going_data.empty else 0.0
        else:
            going_runs, rtr_going_category = 0, 0.0

        meeting_data = historical_data_750[(historical_data_750['jockeyName'] == jockey_name) &
                                            (historical_data_750['meetingName'] == meeting_name)]
        meeting_runs = len(meeting_data['run_to_rating'].dropna()) if not meeting_data.empty else 0
        rtr_meeting = meeting_data['run_to_rating'].dropna().mean() if not meeting_data.empty else 0.0

        tj_data = historical_data_750[(historical_data_750['trainerName'] == trainer_name) &
                                       (historical_data_750['jockeyName'] == jockey_name)]
        trainer_jockey_runs = len(tj_data['run_to_rating'].dropna()) if not tj_data.empty else 0
        rtr_trainer_jockey = tj_data['run_to_rating'].dropna().mean() if not tj_data.empty else 0.0

        horse_metrics.append({
            'horseId': horse_id, 'jockeyName': jockey_name, 'runs': runs,
            'rtr': rtr_formatted, 'rtr_last20': rtr_last20,
            'rtr_going_category': rtr_going_category, 'rtr_meeting': rtr_meeting,
            'rtr_trainer_jockey': rtr_trainer_jockey,
            'going_runs': going_runs, 'meeting_runs': meeting_runs,
            'trainer_jockey_runs': trainer_jockey_runs
        })

    hm = pd.DataFrame(horse_metrics)
    hm['rtr_last20'] = np.round(hm['rtr_last20'].fillna(0.0), 2)
    for col, run_col in [('rtr_going_category', 'going_runs'), ('rtr_meeting', 'meeting_runs'),
                          ('rtr_trainer_jockey', 'trainer_jockey_runs')]:
        hm[col] = np.round(hm[col].fillna(0.0), 2).astype(str) + ' (' + hm[run_col].astype(str) + ')'

    out = (per_horse_today
           .merge(hm[['horseId', 'runs', 'rtr', 'rtr_last20', 'rtr_going_category',
                       'rtr_meeting', 'rtr_trainer_jockey']], on='horseId', how='left')
           .fillna({'runs': 0, 'rtr': '0.0 (0)', 'rtr_last20': 0.0, 'rtr_going_category': '0.0 (0)',
                    'rtr_meeting': '0.0 (0)', 'rtr_trainer_jockey': '0.0 (0)'}))
    out = out[['horseId', 'jockeyName', 'rtr', 'rtr_last20', 'rtr_going_category',
               'rtr_meeting', 'rtr_trainer_jockey']].sort_values('rtr', ascending=False)
    out['rtr'] = out['rtr'].astype(str)
    return out


def summarize_sire_stats_by_race(runners_tdy_df, races_tdy_df, runners_df, race):
    race_id = runners_tdy_df.raceId.unique()[race]
    if races_tdy_df is not None:
        race_date = pd.to_datetime(races_tdy_df.loc[races_tdy_df['id_race'] == race_id, 'date'].iloc[0])
    else:
        raise ValueError("races_tdy_df is required")

    race_info_row = runners_tdy_df.loc[runners_tdy_df['raceId'] == race_id, ['going_category', 'distance_group']].iloc[0]
    going_category = race_info_row['going_category']
    distance_group = race_info_row['distance_group']

    per_horse_today = (runners_tdy_df.loc[runners_tdy_df['raceId'] == race_id,
                                         ['horseId', 'horseSir', 'trainerName']]
                      .drop_duplicates())

    hist = runners_df.copy()
    hist['date'] = pd.to_datetime(hist['date'], errors='coerce')
    hist = hist.sort_values(['horseId', 'date'])
    hist['run_to_rating'] = (hist.groupby('horseId', group_keys=False)['rating_after_race']
                               .apply(lambda s: s.diff()))
    historical_data = hist[hist['date'] < race_date].copy()

    horse_metrics = []
    for _, horse_row in per_horse_today.iterrows():
        horse_id = horse_row['horseId']
        sire_name = horse_row['horseSir']
        trainer_name = horse_row['trainerName']

        sire_hist = historical_data[historical_data['horseSir'] == sire_name].copy()
        if sire_hist.empty:
            runs, rtr = 0, 0.0
        else:
            runs = len(sire_hist)
            rtr = sire_hist['run_to_rating'].dropna().mean()
            if pd.isna(rtr): rtr = 0.0
        rtr_formatted = str(np.round(rtr, 2)) + ' (' + str(runs) + ')'

        if going_category is not None:
            going_data = historical_data[(historical_data['horseSir'] == sire_name) &
                                          (historical_data['going_category'] == going_category)]
            going_runs = len(going_data['run_to_rating'].dropna()) if not going_data.empty else 0
            rtr_going_category = going_data['run_to_rating'].dropna().mean() if not going_data.empty else 0.0
        else:
            going_runs, rtr_going_category = 0, 0.0

        if distance_group is not None:
            dist_data = historical_data[(historical_data['horseSir'] == sire_name) &
                                         (historical_data['distance_group'] == distance_group)]
            distance_runs = len(dist_data['run_to_rating'].dropna()) if not dist_data.empty else 0
            rtr_distance_group = dist_data['run_to_rating'].dropna().mean() if not dist_data.empty else 0.0
        else:
            distance_runs, rtr_distance_group = 0, 0.0

        ts_data = historical_data[(historical_data['trainerName'] == trainer_name) &
                                   (historical_data['horseSir'] == sire_name)]
        trainer_sire_runs = len(ts_data['run_to_rating'].dropna()) if not ts_data.empty else 0
        rtr_trainer_sire = ts_data['run_to_rating'].dropna().mean() if not ts_data.empty else 0.0

        horse_metrics.append({
            'horseId': horse_id, 'horseSir': sire_name, 'runs': runs,
            'rtr': rtr_formatted, 'rtr_going_category': rtr_going_category,
            'rtr_distance_group': rtr_distance_group, 'rtr_trainer_sire': rtr_trainer_sire,
            'going_runs': going_runs, 'distance_runs': distance_runs, 'trainer_sire_runs': trainer_sire_runs
        })

    hm = pd.DataFrame(horse_metrics)
    for col, run_col in [('rtr_going_category', 'going_runs'), ('rtr_distance_group', 'distance_runs'),
                          ('rtr_trainer_sire', 'trainer_sire_runs')]:
        hm[col] = np.round(hm[col].fillna(0.0), 2).astype(str) + ' (' + hm[run_col].astype(str) + ')'

    out = (per_horse_today
           .merge(hm[['horseId', 'runs', 'rtr', 'rtr_going_category',
                       'rtr_distance_group', 'rtr_trainer_sire']], on='horseId', how='left')
           .fillna({'runs': 0, 'rtr': '0.0 (0)', 'rtr_going_category': '0.0 (0)',
                    'rtr_distance_group': '0.0 (0)', 'rtr_trainer_sire': '0.0 (0)'}))
    out = out[['horseId', 'horseSir', 'rtr', 'rtr_going_category',
               'rtr_distance_group', 'rtr_trainer_sire']].sort_values('rtr', ascending=False)
    out['rtr'] = out['rtr'].astype(str)
    return out


def summarize_draw_stats_by_race(runners_tdy_df, races_tdy_df, runners_df, race):
    race_id = runners_tdy_df.raceId.unique()[race]
    if races_tdy_df is not None:
        race_date = pd.to_datetime(races_tdy_df.loc[races_tdy_df['id_race'] == race_id, 'date'].iloc[0])
    else:
        raise ValueError("races_tdy_df is required")

    cutoff_date_750 = race_date - pd.Timedelta(days=9999)

    per_horse_today = (runners_tdy_df.loc[runners_tdy_df['raceId'] == race_id,
                                         ['horseId', 'draw_unique']]
                      .drop_duplicates())
    per_horse_today['draw'] = per_horse_today['draw_unique'].str.split(' - ').str[0]

    hist = runners_df.copy()
    hist['date'] = pd.to_datetime(hist['date'], errors='coerce')
    hist = hist.sort_values(['horseId', 'date'])
    hist['run_to_rating'] = (hist.groupby('horseId', group_keys=False)['rating_after_race']
                               .apply(lambda s: s.diff()))
    historical_data_750 = hist[(hist['date'] >= cutoff_date_750) & (hist['date'] < race_date)].copy()

    horse_metrics = []
    for _, horse_row in per_horse_today.iterrows():
        horse_id = horse_row['horseId']
        draw_unique = horse_row['draw_unique']
        draw = horse_row['draw']

        draw_data = historical_data_750[historical_data_750['draw_unique'] == draw_unique]
        draw_runs = len(draw_data['run_to_rating'].dropna()) if not draw_data.empty else 0
        rtr = draw_data['run_to_rating'].dropna().mean() if not draw_data.empty else 0.0

        horse_metrics.append({'horseId': horse_id, 'draw': draw, 'rtr': rtr, 'draw_runs': draw_runs})

    hm = pd.DataFrame(horse_metrics)
    hm['rtr'] = np.round(hm['rtr'].fillna(0.0), 2).astype(str) + ' (' + hm['draw_runs'].astype(str) + ')'

    out = (per_horse_today
           .merge(hm[['horseId', 'rtr']], on='horseId', how='left')
           .fillna({'rtr': '0.0 (0)'}))
    out = out[['horseId', 'draw', 'rtr']].sort_values('rtr', ascending=False)
    out['rtr'] = out['rtr'].astype(str)
    return out

## 11. Generate Text Files & CSVs

In [ ]:
# ── Main loop: generate text files & CSVs for all today's races ───────────────

DRIVE_BASE = BASE
all_race_ids = runners_tdy.raceId.unique()
print(f"Found {len(all_race_ids)} races to process...")

collectors = {
    'race_info':     [],
    'form_tables':   [],
    'horse_stats':   [],
    'trainer_stats': [],
    'jockey_stats':  [],
    'sire_stats':    [],
    'draw_stats':    [],
}

for race_index, race_id in enumerate(all_race_ids):
    try:
        race_info = (
            races_tdy[races_tdy['id_race'] == race_id]
            [['id_race', 'date', 'name_meeting', 'class', 'type',
              'distance', 'totalPrize', 'minAge', 'maxAge']]
            .merge(runners_tdy[['raceId', 'going_category']],
                   left_on='id_race', right_on='raceId')
            [['date', 'name_meeting', 'class', 'type', 'distance',
              'going_category', 'totalPrize', 'minAge', 'maxAge']]
            .drop_duplicates()
        )
        horse_stats   = summarize_horse_stats_by_race(runners_tdy, races_tdy, df_with_ratings, race_index)
        form_table    = get_horse_form_table(runners_tdy, races_tdy, df_with_ratings, race_index)
        trainer_stats = summarize_trainer_stats_by_race(runners_tdy, races_tdy, df_with_ratings, race_index)
        jockey_stats  = summarize_jockey_stats_by_race(runners_tdy, races_tdy, df_with_ratings, race_index)
        sire_stats    = summarize_sire_stats_by_race(runners_tdy, races_tdy, df_with_ratings, race_index)
        draw_stats    = summarize_draw_stats_by_race(runners_tdy, races_tdy, df_with_ratings, race_index)

        # Collect for CSV export (add id cols inline, no extra named copies)
        for key, df in zip(collectors.keys(),
                           [race_info, form_table, horse_stats,
                            trainer_stats, jockey_stats, sire_stats, draw_stats]):
            collectors[key].append(df.assign(race_id=race_id, race_index=race_index + 1))

        # Write text file
        tables = [
            ('RACE_INFO',     race_info),
            ('FORM_TABLE',    form_table),
            ('HORSE_STATS',   horse_stats),
            ('TRAINER_STATS', trainer_stats),
            ('JOCKEY_STATS',  jockey_stats),
            ('SIRE_STATS',    sire_stats),
            ('DRAW_STATS',    draw_stats),
        ]
        out_path = f"{DRIVE_BASE}/race{race_index + 1}.txt"
        with open(out_path, "w", encoding="utf-8") as f:
            f.write(f"RACE {race_index + 1} (ID: {race_id})\n")
            f.write("=" * 50 + "\n\n")
            for table_name, table_df in tables:
                f.write(f"======================================\n {table_name}\n======================================\n\n")
                f.write(table_df.to_markdown(index=False, tablefmt="github"))
                f.write("\n\n\n")
        print(f"✅ Race {race_index + 1} (ID: {race_id}) → {out_path}")

    except Exception as e:
        print(f"❌ Race {race_index + 1} (ID: {race_id}): {e}")

print(f"\n🏁 Finished processing {len(all_race_ids)} races!")

# ── Save consolidated CSVs ────────────────────────────────────────────────────
print("\n📊 Saving consolidated CSVs...")
for table_name, data_list in collectors.items():
    if data_list:
        try:
            combined = pd.concat(data_list, ignore_index=True)
            csv_path = f"{DRIVE_BASE}/{table_name}_all_races.csv"
            combined.to_csv(csv_path, index=False, encoding="utf-8")
            print(f"✅ {table_name.upper()}: {len(combined)} rows → {csv_path}")
        except Exception as e:
            print(f"❌ {table_name}: {e}")

print("\n🎯 Complete!")


Found 7 races to process...
✅ Race 1 (ID: 1632043) → /content/drive/MyDrive/PT/race1.txt
✅ Race 2 (ID: 1632042) → /content/drive/MyDrive/PT/race2.txt
✅ Race 3 (ID: 1632039) → /content/drive/MyDrive/PT/race3.txt
✅ Race 4 (ID: 1632038) → /content/drive/MyDrive/PT/race4.txt
✅ Race 5 (ID: 1632036) → /content/drive/MyDrive/PT/race5.txt
✅ Race 6 (ID: 1632037) → /content/drive/MyDrive/PT/race6.txt
✅ Race 7 (ID: 1632040) → /content/drive/MyDrive/PT/race7.txt

🏁 Finished processing 7 races!

📊 Saving consolidated CSVs...
✅ RACE_INFO: 7 rows → /content/drive/MyDrive/PT/race_info_all_races.csv
✅ FORM_TABLES: 2133 rows → /content/drive/MyDrive/PT/form_tables_all_races.csv
✅ HORSE_STATS: 95 rows → /content/drive/MyDrive/PT/horse_stats_all_races.csv
✅ TRAINER_STATS: 95 rows → /content/drive/MyDrive/PT/trainer_stats_all_races.csv
✅ JOCKEY_STATS: 95 rows → /content/drive/MyDrive/PT/jockey_stats_all_races.csv
✅ SIRE_STATS: 95 rows → /content/drive/MyDrive/PT/sire_stats_all_races.csv
✅ DRAW_STATS: 95 ro